# <center> VAI Store - Engenharia de Variáveis </center>

---

Com base nos **_insights_** da Análise Exploratória de Dados, nosso objetivo nesta etapa é **traduzir os padrões que descobrimos** (como sazonalidade e diferença entre filiais) em _**features**_ que o modelo possa usar para prever a demanda dos produtos.

### **1. Configuração e Imports**

Visão Geral: Vamos começar importando as bibliotecas essenciais (pandas para manipulação de dados e numpy para operações numéricas). Também definiremos os caminhos relativos para os arquivos de dados brutos, facilitando a organização.

In [1]:
import pandas as pd
import numpy as np
import os

# Define os caminhos relativos para "subir um nível" (../)
# e depois entrar na pasta 'data'
RAW_DATA_PATH = os.path.join('..', 'data', 'raw')
VENDAS_FILE = os.path.join(RAW_DATA_PATH, 'vendas.csv')
PRODUTO_FILE = os.path.join(RAW_DATA_PATH, 'produto.csv')

# Define o caminho de saída da mesma forma
PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed')

# O arquivo de saída agora é .parquet
OUTPUT_FILE = os.path.join(PROCESSED_DATA_PATH, 'feature_engineered_dataset.parquet')

# Garante que o diretório de saída exista no local correto
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Imprime os caminhos absolutos para verificação
print(f"Lendo dados de: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Salvando dados em: {os.path.abspath(PROCESSED_DATA_PATH)}")

Lendo dados de: c:\Users\felip\OneDrive\Documentos\GitHub\Grupo6-ProjetoFinal\data\raw
Salvando dados em: c:\Users\felip\OneDrive\Documentos\GitHub\Grupo6-ProjetoFinal\data\processed


### **2. Funções Auxiliares de Carga e Limpeza**

Visão Geral: Para manter o notebook limpo, definiremos duas funções auxiliares. A load_csv lida com os diferentes tipos de codificação (utf-8 ou latin1) que podemos encontrar. A clean_sku padroniza a coluna SKU, que observamos ter espaços extras e tipo de dado incorreto, garantindo que as chaves de merge sejam consistentes.

In [2]:
def clean_sku(sku_series):
    """Limpa a coluna SKU, removendo espaços e convertendo para inteiro."""
    return pd.to_numeric(sku_series.astype(str).str.strip(), errors='coerce').astype('Int64')

def load_csv(file_path):
    """Tenta carregar um CSV com encoding 'utf-8', e usa 'latin1' como fallback."""
    try:
        return pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='latin1')

### **3. Carregar e Agregar Dados de Vendas**

Visão Geral: Esta etapa carrega os dados brutos de vendas.csv. A principal transformação é a agregação. Como os dados originais são transacionais (múltiplas vendas por dia), eles são consolidados para refletir a performance total por produto.

In [3]:
df_vendas = load_csv(VENDAS_FILE)

# Limpezas básicas
df_vendas['DATA_ATEND'] = pd.to_datetime(df_vendas['DATA_ATEND'])
df_vendas['SKU'] = clean_sku(df_vendas['SKU'])

# Remove SKUs nulos e vendas com faturamento zero (não contribuem)
df_vendas = df_vendas.dropna(subset=['SKU'])
df_vendas = df_vendas[df_vendas['FATUR_VENDA'] > 0]

print(f"Dados transacionais de vendas carregados. Shape: {df_vendas.shape}")

Dados transacionais de vendas carregados. Shape: (556198, 10)


### **4. Adicionar Contexto do Produto (Categorias)**

Visão Geral: Para agregar por CATEGORIA, primeiro precisamos de saber a qual categoria cada SKU (produto) pertence. Esta etapa carrega o produto.csv, limpa os dados de categoria, e faz um merge com a tabela transacional de vendas. O resultado é que cada linha de venda agora "sabe" a sua respetiva CATEGORIA.

In [4]:
# Carrega e limpa os dados de produtos
df_prod = load_csv(PRODUTO_FILE)
df_prod['SKU'] = clean_sku(df_prod['SKU'])
df_prod = df_prod.dropna(subset=['SKU'])

# Seleciona, limpa e padroniza as colunas de contexto
# Mantemos a Subcategoria por enquanto, para a agregação opcional
df_prod_context = df_prod[['SKU', 'CATEGORIA', 'SUBCATEGORIA']].copy()
df_prod_context['CATEGORIA'] = df_prod_context['CATEGORIA'].fillna('DESCONHECIDA')
df_prod_context['SUBCATEGORIA'] = df_prod_context['SUBCATEGORIA'].fillna('DESCONHECIDA')

# Garante uma chave única
df_prod_context = df_prod_context.drop_duplicates(subset=['SKU'])

# Mescla o contexto ao dataset transacional de vendas
df_vendas_com_categoria = pd.merge(df_vendas, df_prod_context, on='SKU', how='left')
df_vendas_com_categoria[['CATEGORIA', 'SUBCATEGORIA']] = df_vendas_com_categoria[['CATEGORIA', 'SUBCATEGORIA']].fillna('DESCONHECIDA')

print(f"Contexto do produto mesclado às transações. Shape: {df_vendas_com_categoria.shape}")

Contexto do produto mesclado às transações. Shape: (556198, 12)


### **5. Agregar Faturamento por Categoria**

Visão Geral: Seguindo a dica do case ("não fazer previsão por produto"), agora agregamos o nosso dataset. Agrupamos os dados por DATA_ATEND e CATEGORIA e somamos o FATUR_VENDA. O resultado é um dataset que mostra o faturamento total de cada categoria por dia (ex: Faturamento de 'Bacalhau & Pescados' em 2024-12-10).

In [5]:
# Agrega por Data e Categoria
df_agg_categoria = df_vendas_com_categoria.groupby(['DATA_ATEND', 'CATEGORIA']).agg(
    FATUR_TOTAL=('FATUR_VENDA', 'sum')
).reset_index()

print(f"Dados agregados por Categoria. Shape: {df_agg_categoria.shape}")

Dados agregados por Categoria. Shape: (4689, 3)


### **6. Criar "Scaffold" (Grid de Datas Completo)**

Visão Geral: Os dados agregados por categoria ainda são "esparsos"; pode haver dias em que uma categoria inteira não teve vendas. Para garantir que o modelo aprenda com os dias de faturamento zero, esta etapa cria um "scaffold" (gabarito denso). O scaffold contém uma linha para cada dia (do início ao fim do período) e para cada CATEGORIA. Os dados agregados são, então, mesclados a este gabarito.

In [6]:
# A agregação agora é por Categoria
group_cols = ['CATEGORIA']

# Encontra o range de datas e os grupos (Categorias) únicos
min_date = df_agg_categoria['DATA_ATEND'].min()
max_date = df_agg_categoria['DATA_ATEND'].max()
date_range = pd.date_range(min_date, max_date, freq='D')
unique_groups = df_agg_categoria[group_cols].drop_duplicates()

# Cria o scaffold
df_scaffold = pd.MultiIndex.from_product(
    [unique_groups[col] for col in group_cols] + [date_range],
    names=group_cols + ['DATA_ATEND']
).to_frame(index=False)

# Junta os dados reais ao scaffold
df_full = pd.merge(df_scaffold, df_agg_categoria, on=group_cols + ['DATA_ATEND'], how='left')

print(f"Scaffold criado. Shape: {df_full.shape}")

Scaffold criado. Shape: (4745, 3)


### **7. Preencher Dados Ausentes (Pós-Scaffold)**

Visão Geral: Após a mesclagem com o scaffold, os dias em que não houve faturamento para uma categoria existem como linhas com valores NaN (nulos). Esta etapa trata esses nulos:

* O FATUR_TOTAL é preenchido com 0, ensinando ao modelo que naqueles dias não houve receita para aquela categoria.

* A coluna CATEGORIA é preenchida usando ffill (propagação do último valor válido) para garantir que esteja presente em todas as linhas do scaffold.

In [7]:
# Ordena para garantir que o ffill funcione corretamente
df_full = df_full.sort_values(by=['CATEGORIA', 'DATA_ATEND'])

# Preenche métricas com 0
df_full['FATUR_TOTAL'] = df_full['FATUR_TOTAL'].fillna(0)

# Preenche contexto (agrupado por Categoria)
context_cols = ['CATEGORIA']
df_full[context_cols] = df_full.groupby(group_cols)[context_cols].ffill().bfill()

# Remove linhas que não tiveram contexto (raro)
df_full = df_full.dropna(subset=context_cols)

print("Preenchimento concluído.")

Preenchimento concluído.


### **8. Engenharia de Features de Calendário**

Visão Geral: Para capturar a sazonalidade, o modelo precisa de features numéricas que representem o tempo. Esta etapa extrai diversas features da DATA_ATEND, incluindo:

* Ciclos Curtos: DIA_SEMANA, DIA_DO_MES.

* Ciclos Longos: DIA_DO_ANO, MES, ANO, SEMANA_DO_ANO.

* Eventos de Negócio: INICIO_MES e FIM_MES (capturando os picos de "dia de pagamento").

In [8]:
date_col = df_full['DATA_ATEND']

df_full['DIA_SEMANA'] = date_col.dt.dayofweek
df_full['DIA_DO_MES'] = date_col.dt.day
df_full['DIA_DO_ANO'] = date_col.dt.dayofyear
df_full['MES'] = date_col.dt.month
df_full['SEMANA_DO_ANO'] = date_col.dt.isocalendar().week.astype(int)
df_full['ANO'] = date_col.dt.year

df_full['INICIO_MES'] = (date_col.dt.day <= 5).astype(int)
df_full['FIM_MES'] = (date_col.dt.day >= 28).astype(int)

print("Features de calendário criadas.")

Features de calendário criadas.


### **9. Engenharia de Features de Feriados**

Visão Geral: Com base na análise exploratória (que mostrou picos em feriados), esta etapa codifica os eventos sazonais mais importantes. Usando a biblioteca holidays, são criadas flags binárias para EH_FERIADO e, o mais importante, janelas de antecipação para os principais eventos de vendas: ANTECIPACAO_PASCOA_14D, ANTECIPACAO_NATAL_21D e ANTECIPACAO_ANO_NOVO_7D.

In [9]:
import holidays

# Pega os anos únicos do nosso dataset (que já está no df_full)
anos = df_full['ANO'].unique()

# Cria um objeto de feriados do Brasil para os anos relevantes
br_holidays = holidays.Brazil(years=anos)

# Converte o dicionário de feriados em um DataFrame
df_holidays = pd.DataFrame(br_holidays.items(), columns=['DATA_ATEND', 'NOME_FERIADO'])
df_holidays['DATA_ATEND'] = pd.to_datetime(df_holidays['DATA_ATEND'])
df_holidays['EH_FERIADO'] = 1

pascoa_datas = {}
for date, name in br_holidays.items():
    if name == 'Páscoa':
        pascoa_datas[date.year] = pd.to_datetime(date)
natal_datas = {ano: pd.to_datetime(f'{ano}-12-25') for ano in anos}
anos_com_seguinte = np.append(anos, anos.max() + 1)
ano_novo_datas = {ano: pd.to_datetime(f'{ano+1}-01-01') for ano in anos_com_seguinte}

df_full['DATA_PASCOA'] = df_full['ANO'].map(pascoa_datas)
df_full['DATA_NATAL'] = df_full['ANO'].map(natal_datas)
df_full['DATA_ANO_NOVO'] = df_full['ANO'].map(ano_novo_datas)

# Força a conversão de tipo ANTES da subtração
data_pascoa_ts = pd.to_datetime(df_full['DATA_PASCOA'])
data_natal_ts = pd.to_datetime(df_full['DATA_NATAL'])
data_ano_novo_ts = pd.to_datetime(df_full['DATA_ANO_NOVO'])
data_atend_ts = pd.to_datetime(df_full['DATA_ATEND'])

# Calcula os dias restantes até esses eventos
df_full['DIAS_ATE_PASCOA'] = (data_pascoa_ts - data_atend_ts).dt.days
df_full['DIAS_ATE_NATAL'] = (data_natal_ts - data_atend_ts).dt.days
df_full['DIAS_ATE_ANO_NOVO'] = (data_ano_novo_ts - data_atend_ts).dt.days

# Criando as features da sua lista
df_full['ANTECIPACAO_PASCOA_14D'] = ((df_full['DIAS_ATE_PASCOA'] >= 0) & (df_full['DIAS_ATE_PASCOA'] <= 14)).astype(int)
df_full['ANTECIPACAO_NATAL_21D'] = ((df_full['DIAS_ATE_NATAL'] >= 0) & (df_full['DIAS_ATE_NATAL'] <= 21)).astype(int)
df_full['ANTECIPACAO_ANO_NOVO_7D'] = ((df_full['DIAS_ATE_ANO_NOVO'] >= 0) & (df_full['DIAS_ATE_ANO_NOVO'] <= 7)).astype(int)

# Junta a feature EH_FERIADO
df_full = pd.merge(df_full, df_holidays[['DATA_ATEND', 'EH_FERIADO']], on='DATA_ATEND', how='left')
df_full['EH_FERIADO'] = df_full['EH_FERIADO'].fillna(0).astype(int)

# Limpa colunas auxiliares
df_full = df_full.drop(columns=[
    'DATA_PASCOA', 'DATA_NATAL', 'DATA_ANO_NOVO',
    'DIAS_ATE_PASCOA', 'DIAS_ATE_NATAL', 'DIAS_ATE_ANO_NOVO'
])
df_full = df_full.fillna(0) # Limpa NaNs de eventos

print("Features de feriados criadas.")

Features de feriados criadas.


### **10. Engenharia de Features de Lag (Defasagem)**

Visão Geral: Para dar "memória" ao modelo, criamos features de lag (defasagem). Estamos focando no histórico de curto prazo (1, 2 e 3 dias atrás) e na sazonalidade semanal (7 e 14 dias atrás). O alvo para os lags é o FATUR_TOTAL, garantindo que o modelo aprenda a partir do histórico de receita do produto.

In [10]:
# Define os lags de curto prazo e sazonais
lags = [1, 2, 3, 7, 14]
target_col = 'FATUR_TOTAL'
group_cols = ['CATEGORIA'] # A agregação agora é por Categoria

# Ordena os dados para garantir que o shift seja temporalmente correto
df_full = df_full.sort_values(by=['CATEGORIA', 'DATA_ATEND'])

for lag in lags:
    col_name = f'{target_col}_LAG_{lag}'
    df_full[col_name] = df_full.groupby(group_cols)[target_col].shift(lag)

# Os lags iniciais criarão NaNs, que serão tratados na próxima etapa
print(f"Features de lag {lags} criadas para {target_col}.")

Features de lag [1, 2, 3, 7, 14] criadas para FATUR_TOTAL.


### **11. Engenharia de Features de Interação (Regra de Negócio)**

Visão Geral: A análise de erros (Etapa 7 do notebook 03) mostrou que o modelo "generalista" falhou em aplicar as features de evento (ex: ANTECIPACAO_NATAL_21D) apenas às categorias relevantes (ex: Bacalhau & Pescados), levando a erros graves de superestimação (em Lanchonete) e subestimação (em Bacalhau).

Para resolver isto, esta etapa cria features de interação explícitas. Estas features combinam o evento (ANTECIPACAO_...) com a categoria (CATEGORIA == 'Bacalhau & Pescados'). O modelo agora aprenderá uma regra específica e robusta, que só será ativada para a categoria correta no período de evento correto.

In [11]:
# --- NOVAS FEATURES DE INTERAÇÃO ---
# Esta é a chave: a flag só é 1 se AMBAS as condições forem verdadeiras
# (O nome da categoria "Bacalhau & Pescados" deve ser exato)

df_full['INTERACAO_PASCOA_BACALHAU'] = (
    (df_full['ANTECIPACAO_PASCOA_14D'] == 1) &
    (df_full['CATEGORIA'] == 'Bacalhau & Pescados')
).astype(int)

df_full['INTERACAO_NATAL_BACALHAU'] = (
    (df_full['ANTECIPACAO_NATAL_21D'] == 1) &
    (df_full['CATEGORIA'] == 'Bacalhau & Pescados')
).astype(int)

df_full['INTERACAO_ANO_NOVO_BACALHAU'] = (
    (df_full['ANTECIPACAO_ANO_NOVO_7D'] == 1) &
    (df_full['CATEGORIA'] == 'Bacalhau & Pescados')
).astype(int)
# --- FIM DAS NOVAS FEATURES ---

# Nota: Podemos adicionar outras interações se necessário (ex: Castanhas no Natal)
print("Features de interação criadas.")

Features de interação criadas.


### **12. Limpeza Final e Salvamento**

Visão Geral: As features de lag e rolling criaram NaNs no início da série temporal de cada produto (o que é normal). Vamos preenchê-los com 0 (assumindo que não há histórico anterior) e salvar o dataset final e pronto para o modelo no caminho data/processed que definimos no início.

In [12]:
print("Iniciando limpeza final e salvamento...")

# Preenche os NaNs dos lags (e qualquer outro) com 0
df_final = df_full.fillna(0)

# Opcional: Converte tipos de dados para otimizar uso de memória
for col in df_final.columns:
    if col.startswith(('FATUR_TOTAL')): # Pega o alvo e os novos lags
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='float')
    
    # ATUALIZADO: Inclui 'INTERACAO_' na otimização
    if col.startswith(('DIA_', 'MES', 'ANO', 'SEMANA_', 'INICIO_', 'FIM_', 'EH_FERIADO', 'ANTECIPACAO_', 'INTERACAO_')):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='integer')

# Salva o arquivo em Parquet
print(f"Salvando arquivo em formato Parquet em: {OUTPUT_FILE}")
df_final.to_parquet(OUTPUT_FILE, index=False, engine='pyarrow')

print(f"\nEngenharia de features (v2 com Interações) concluída! Dataset salvo.")
print("\n--- Informações do DataFrame Final ---")
df_final.info()

Iniciando limpeza final e salvamento...
Salvando arquivo em formato Parquet em: ..\data\processed\feature_engineered_dataset.parquet

Engenharia de features (v2 com Interações) concluída! Dataset salvo.

--- Informações do DataFrame Final ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4745 entries, 0 to 4744
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   CATEGORIA                    4745 non-null   object        
 1   DATA_ATEND                   4745 non-null   datetime64[ns]
 2   FATUR_TOTAL                  4745 non-null   float64       
 3   DIA_SEMANA                   4745 non-null   int8          
 4   DIA_DO_MES                   4745 non-null   int8          
 5   DIA_DO_ANO                   4745 non-null   int16         
 6   MES                          4745 non-null   int8          
 7   SEMANA_DO_ANO                4745 non-null   int8          


In [13]:
print("Iniciando limpeza final e salvamento...")

# Preenche os NaNs dos lags (e qualquer outro) com 0
df_final = df_full.fillna(0)

# Opcional: Converte tipos de dados para otimizar uso de memória
for col in df_final.columns:
    
    # Usar .astype('float32') é mais robusto que downcast='float'
    if col.startswith(('FATUR_TOTAL')): # Pega o alvo e os novos lags
        df_final[col] = df_final[col].astype('float32')
    
    if col.startswith(('DIA_', 'MES', 'ANO', 'SEMANA_', 'INICIO_', 'FIM_', 'EH_FERIADO', 'ANTECIPACAO_', 'INTERACAO_')):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='integer')

# Salva o arquivo em Parquet
print(f"Salvando arquivo em formato Parquet em: {OUTPUT_FILE}")
df_final.to_parquet(OUTPUT_FILE, index=False, engine='pyarrow')

print(f"\nEngenharia de features (v2 com Interações) concluída! Dataset salvo.")
print("\n--- Informações do DataFrame Final ---")
df_final.info()

Iniciando limpeza final e salvamento...
Salvando arquivo em formato Parquet em: ..\data\processed\feature_engineered_dataset.parquet

Engenharia de features (v2 com Interações) concluída! Dataset salvo.

--- Informações do DataFrame Final ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4745 entries, 0 to 4744
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   CATEGORIA                    4745 non-null   object        
 1   DATA_ATEND                   4745 non-null   datetime64[ns]
 2   FATUR_TOTAL                  4745 non-null   float32       
 3   DIA_SEMANA                   4745 non-null   int8          
 4   DIA_DO_MES                   4745 non-null   int8          
 5   DIA_DO_ANO                   4745 non-null   int16         
 6   MES                          4745 non-null   int8          
 7   SEMANA_DO_ANO                4745 non-null   int8          
